# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *70* |
| **Integrantes** | *Isabela Cuartas Vence · Juan Camilo Gomez Murillo* |
| **Caso de estudio** | *(Wanderbricks u otro)* |
| **Fecha de entrega** | domingo 23 de agosto |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*¿Qué se quiere resolver y por qué importa? Máximo tres párrafos.
Debe quedar claro qué preguntas del negocio deberá responder esta base de datos.*

---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

In [0]:
# TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)
#EXPLICACIÓN:La exploración reveló un problema de calidad concreto: booking_updates presenta 7,943 valores duplicados en su llave booking_update_id (75,125 distintos sobre 83,068 filas totales), lo que exige deduplicación explícita antes de construir la capa plata. Asimismo, la cardinalidad de user_id en bookings (54,708 sobre 72,247 reservas) confirma una relación uno-a-muchos típica del dominio transaccional, mientras que columnas como status (solo 4 valores distintos) son claramente categóricas y no llaves.
from pyspark.sql import functions as F

tablas = ["users", "countries", "bookings", "booking_updates"]

for t in tablas:
    print(f"\n--- Nulos en {t} ---")
    df = spark.table(f"samples.wanderbricks.{t}")
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

#Cardinalidades (columna por columna, para cada tabla):

for t in tablas:
    print(f"\n--- Cardinalidad en {t} ---")
    df = spark.table(f"samples.wanderbricks.{t}")
    total = df.count()
    df.select([F.countDistinct(F.col(c)).alias(c) for c in df.columns]).display()
    print(f"(Total de filas para comparar: {total:,})")

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*



Criterio del caso: Relacion users - bookings (1 usuario -> varias reservas, cardinalidad 54708 usuarios / 72247 reservas)  

Relacional: Queda exacto con las llaves foraneas nativas para este tipo de relacion de 1 a muchos 

NOSQL: Se tendria que hacer un duplicado entre cada reserva o pasar directamente a hacer unas busquedas manuales

Lakehouse: Es igual a una base de datos relacional, se hace via JOIN sobre unas tablas DELTA 

Decision: Lakehouse tiene un mismo beneficio, sin tener que perder una flexibilidad futura

____________________________________________________________________________________________________________________________

Criterio del caso: Integridad de llaves (booking_id y user_id no tiene duplicados resaltados en la seccion 2)

Relacional: constraints y llaves primarias que garantizan una unidad 

NOSQL: La mayoria de NOSQL no valida unidad de forma nativa

Lakehouse: Se puede confirmar con MERGE / dropduplicates, aunque no hay un constraint automatizado como en relacional exacto

Decision: Relacional gana en este ambito puntual, pero lakehouse, mejora muchisimo mas con la compensa con control programatico

____________________________________________________________________________________________________________________________

Criterio del caso: Confirmacion del historial de cambios (7943 duplicados que han sido detectados en booking_updates y evolucionan en estados de reserva)

Relacional: No existe forma nativa de ir a una version anterior de la tabla sin diseñar una auditoria manual

NOSQL: No tiene una version nativamente

Lakehouse: "DESCRIBE HISTORY" y "VERSION AS OF" sin diseño adicional

Decision: Lakeouse (Ventaja abismal)

____________________________________________________________________________________________________________________________

Criterio del caso: Transacciones atomicas al actualizar reservas (MERGE de "booking_updates" hacia "bookings")

Relacional: Soporta ACID de forma efectiva con los motores que tiene como PostgreSQL

NOSQL: La mayoria de NoSQL sacrifica a ACID por escalabilidad (Exceptuando casos muy especificos)

Lakehouse: ACID completo via DeltaLake, igual de relleno y robusto que el relacional

Decision: Diriamos que es un empate puesto que entre relacional y lakehouse, gana lakehouse por sumar todo lo anterior

____________________________________________________________________________________________________________________________

Criterio del caso: Escalabilidad y datos futuros mixtos

RelacionaL: Agregar esa tabla obligaria a tener que normalizar los arrays en multiples tablas entrelazadas

NOSQL: En ese ambito es flexible pero perderia la consistencia que ya tienen sus tablas transaccionales

Lakehouse: Combina la flexibilidad para clickstream sin perder ACID en bookings y users

Decision: Lakehouse (Por motivos principales)

____________________________________________________________________________________________________________________________

Explicacion:

Si bien las cuatro tablas examinadas son completamente tabulares y encajarían en un modelo relacional puro, el caso de Wanderbricks en su totalidad presenta datos de navegación con estructuras anidadas que un modelo relacional podría manejar con dificultad.
Por eso se optó por un lakehouse, conserva las garantías de integridad y las relaciones llave-foránea del modelo relacional , pero añade capacidades de versionado  y flexibilidad para datos semiestructurados que el proyecto necesitará en fases posteriores.

**Referencias (APA 7):**
Armbrust, M., Ghodsi, A., Xin, R., & Zaharia, M. (2021). 
Databricks. (2024). 
    

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupoNN"   # TODO: reemplazar NN
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
# TODO: ingerir las tablas del caso a la capa bronce, con esquema explícito

### 4.3 Capa plata — datos limpios y tipados

In [0]:
# TODO: limpieza, tipado y reglas de negocio

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
# TODO: leer y aplanar estructuras anidadas

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
# ACID — una operación que modifique datos
# TODO

In [0]:
# Time travel
# display(spark.sql(f"DESCRIBE HISTORY {TABLA}"))
# TODO: consultar una versión anterior y comparar

In [0]:
# Evolución de esquema con mergeSchema
# TODO

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

In [0]:
# Consulta 1 — pregunta que responde:
# TODO

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas